# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [11]:
import pandas as pd
import plotly.express as px
import numpy as np

In [12]:
import micropip
await micropip.install("plotly")

---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


In [14]:
# Load the dataset
df = pd.read_csv('world_population.csv')

# Rename columns
df = df.rename(columns={
    'Country/Territory': 'Country',
    'Area (km²)': 'Area',
    'Density (per km²)': 'Density',
    'World Population Percentage': 'World share (%)'
})

# Create long-format dataset
year_cols = [c for c in df.columns if c.endswith('Population')]

df_long = pd.melt(
    df,
    id_vars=['CCA3', 'Country', 'Continent'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Population'
)

df_long['Year'] = df_long['Year'].str[:4].astype(int)

print(df.shape, df_long.shape)

(234, 16) (1872, 5)


In [15]:
# Q1
growth = df.groupby('Continent').agg(
    Population_1970=('1970 Population', 'sum'),
    Population_2022=('2022 Population', 'sum')
)

growth['Growth (%)'] = (
    (growth['Population_2022'] - growth['Population_1970'])
    / growth['Population_1970'] * 100
)

growth = growth.sort_values('Growth (%)', ascending=False)

print(growth)

               Population_1970  Population_2022  Growth (%)
Continent                                                  
Africa               365444348       1426730932  290.409905
Oceania               19480270         45038554  131.200871
South America        192947156        436816608  126.391835
Asia                2144906290       4721383274  120.120725
North America        315434606        600296136   90.307634
Europe               655923991        743147538   13.297813


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
I chose a horizontal bar chart because it makes it easy to compare and rank the 15 countries. African countries are highlighted in a different colour.

In [20]:
# Q1 - Line chart

continent_trend = df_long.groupby(
    ['Continent', 'Year'], as_index=False
)['Population'].sum()

baseline = continent_trend[
    continent_trend['Year'] == 1970
][['Continent', 'Population']]

baseline = baseline.rename(
    columns={'Population': 'Population_1970'}
)

continent_trend = continent_trend.merge(
    baseline, on='Continent'
)

continent_trend['Growth Index'] = (
    continent_trend['Population']
    / continent_trend['Population_1970'] * 100
)

fig = px.line(
    continent_trend,
    x='Year',
    y='Growth Index',
    color='Continent',
    markers=True,
    title='Population Growth by Continent (1970 = 100)',
    labels={
        'Growth Index': 'Population Growth Index',
        'Year': 'Year'
    },
    color_discrete_sequence=[
        '#0072B2', '#E69F00', '#56B4E9',
        '#CC79A7', '#999999', '#332288'
    ],
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    legend_title='Continent',
    hovermode='x unified'
)

fig.show()

<class 'ValueError'>: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [19]:
display(HTML(
    fig.to_html(
        include_plotlyjs='cdn',
        full_html=False,
        default_height='600px',
        default_width='100%'
    )
))

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
I chose a horizontal bar chart because it makes it easy to compare the 15 countries. African countries are highlighted in blue, while the others are grey.

In [23]:
# Q2 - Population increase

from IPython.display import HTML, display

df['People Added'] = df['2022 Population'] - df['1970 Population']

top15 = df.sort_values('People Added', ascending=False).head(15)

top15['Group'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other countries'
)

print(top15[['Country', 'Continent', 'People Added']])
print("African countries:", (top15['Continent'] == 'Africa').sum())

fig = px.bar(
    top15.sort_values('People Added'),
    x='People Added',
    y='Country',
    color='Group',
    orientation='h',
    title='Top 15 Countries by Population Increase (1970–2022)',
    color_discrete_map={
        'Africa': '#0072B2',
        'Other countries': '#B0B0B0'
    },
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='People Added',
    yaxis_title='Country'
)

display(HTML(fig.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_height='650px',
    default_width='100%'
)))

           Country      Continent  People Added
92           India           Asia     859671872
41           China           Asia     603352887
156       Pakistan           Asia     176533990
149        Nigeria         Africa     162971948
93       Indonesia           Asia     160272945
221  United States  North America     137961517
27          Brazil  South America     118943623
16      Bangladesh           Asia     103644512
63        Ethiopia         Africa      95071678
55        DR Congo         Africa      78858479
163    Philippines           Asia      78123423
131         Mexico  North America      77214819
57           Egypt         Africa      76208117
94            Iran           Asia      60100865
228        Vietnam           Asia      56258007
African countries: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a violin plot because it shows the spread and variation of population density across different continents.

In [24]:
# Q3 - Land area and population

from IPython.display import HTML, display

print("Correlation:",
      df['Area'].corr(df['2022 Population']))

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    size='World share (%)',
    hover_name='Country',
    log_x=True,
    log_y=True,
    title='Land Area vs Population (2022)',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Land Area (km²)',
    yaxis_title='Population (2022)'
)

display(HTML(fig.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_height='650px',
    default_width='100%'
)))

Correlation: 0.4534113215782548


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*I chose a violin plot because it shows the spread and variation of population density within and across continents.

In [25]:
# Q4 - Population density

from IPython.display import HTML, display

fig = px.violin(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    box=True,
    points='outliers',
    log_y=True,
    title='Population Density Across Continents',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Continent',
    yaxis_title='Population Density (people per km²)',
    showlegend=False
)

display(HTML(fig.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_height='650px',
    default_width='100%'
)))

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*I chose a choropleth map because it shows population growth and decline by country. Different colours make it easy to compare percentage changes across the world.

In [26]:
# Q5 - Population growth and decline

from IPython.display import HTML, display

df['Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

fig = px.choropleth(
    df,
    locations='CCA3',
    color='Change (%)',
    hover_name='Country',
    color_continuous_scale='RdYlBu_r',
    color_continuous_midpoint=0,
    title='Population Growth and Decline by Country (2000–2022)',
    projection='natural earth',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    geo=dict(showframe=False, showcoastlines=True),
    coloraxis_colorbar_title='Change (%)'
)

display(HTML(fig.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_height='650px',
    default_width='100%'
)))

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a treemap because it shows the 2022 world population by continent and country in one chart. Larger boxes represent countries with larger populations.

In [27]:
# Q6 - World population distribution

from IPython.display import HTML, display

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_sequence=[
        '#0072B2', '#E69F00', '#999999',
        '#56B4E9', '#CC79A7', '#009E73'
    ],
    title='World Population by Continent and Country (2022)',
    template='plotly_white'
)

fig.update_layout(title_x=0.5)

display(HTML(fig.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_height='650px',
    default_width='100%'
)))

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a 100% stacked bar chart because it clearly shows how each continent's share of the world population changed over time. Africa's share increased, Asia remained almost stable, and Europe's share decreased.

In [28]:

# Q7 - World population shares (1970–2022)

from IPython.display import HTML, display

# Group the continents
data = df_long.copy()
data['Group'] = data['Continent'].where(
    data['Continent'].isin(['Africa', 'Asia', 'Europe']),
    'Rest of the world'
)

# Calculate population shares
shares = data.groupby(
    ['Year', 'Group'], as_index=False
)['Population'].sum()

shares['Share (%)'] = (
    shares['Population'] /
    shares.groupby('Year')['Population'].transform('sum') * 100
)

# Create stacked bar chart
fig = px.bar(
    shares,
    x='Year',
    y='Share (%)',
    color='Group',
    barmode='stack',
    title='World Population Share by Continent (1970–2022)',
    color_discrete_map={
        'Africa': '#0072B2',
        'Asia': '#E69F00',
        'Europe': '#CC79A7',
        'Rest of the world': '#999999'
    },
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Year',
    yaxis_title='Share of World Population (%)',
    yaxis_range=[0, 100],
    legend_title='Continent'
)

fig.update_xaxes(type='category')

display(HTML(fig.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_height='650px',
    default_width='100%'
)))
